# LeNet-5 CNN 板上推理 + SQ-mapping 打包加速 + Profiler

使用 `cim_driver.py`（CIMDriver / CIMModel）+ `golden_model.py` 进行：
1. AXI 连通性测试
2. LeNet-5 单图推理 + bit-exact 逐层验证 (verify=True)
3. SQ-mapping 打包 Conv 加速（Conv1 ~20x, Conv2 ~5x）
4. 延迟分解 profiler（profile=True）
5. 批量推理正确率测试

**前置条件**：
1. 宿主机运行 `python lenet5_quantize.py`，生成 `lenet5_data/`（含 `lenet5_qparams.npz`）
2. 上传到 PYNQ 同目录：`cim_soc.bit` + `.hwh`、`cim_driver.py`、`golden_model.py`、`lenet5_data/`、本 notebook

In [2]:
# ====================================================================
# 1. 导入 + 加载 bitstream
# ====================================================================
import numpy as np
import glob, os, time
from cim_driver import (
    CIMDriver, CIMModel, _make_run_id,
    weight_to_chunks, bias_to_u32,
    _BASE, _MMIO_SIZE,
)
import golden_model as gm

drv = CIMDriver("cim_soc.bit")
print("Overlay loaded, driver ready.")

Overlay loaded, driver ready.


In [3]:
# ====================================================================
# 2. AXI 连通性测试
# ====================================================================
from pynq import MMIO

mmio = MMIO(_BASE, _MMIO_SIZE)
print("=== AXI Connectivity Test ===")
drv.soft_reset()
mmio.write(0x010, 784)
rb = mmio.read(0x010)
print(f"  IN_DIM write=784, readback={rb}  {'PASS' if rb == 784 else 'FAIL'}")
assert rb == 784, "AXI read/write failed!"
print("AXI OK.")

=== AXI Connectivity Test ===
  IN_DIM write=784, readback=784  PASS
AXI OK.


In [5]:
!unzip lenet5_data.zip

Archive:  lenet5_data.zip
   creating: lenet5_data/
  inflating: lenet5_data/fc4_weight_tiles.hex  
  inflating: lenet5_data/conv1_weight_tiles.hex  
  inflating: lenet5_data/quant_params.hex  
  inflating: lenet5_data/conv2_weight_tiles.hex  
  inflating: lenet5_data/fc5_bias.hex  
  inflating: lenet5_data/zero_points.hex  
  inflating: lenet5_data/fc3_bias.hex  
  inflating: lenet5_data/fc4_bias.hex  
   creating: lenet5_data/test_images/
 extracting: lenet5_data/test_images/img_0004_label.txt  
 extracting: lenet5_data/test_images/img_0176_pred.txt  
  inflating: lenet5_data/test_images/img_0116_fc5.hex  
  inflating: lenet5_data/test_images/img_0102.hex  
 extracting: lenet5_data/test_images/img_0145_fc5.hex  
  inflating: lenet5_data/test_images/img_0037_fc5.hex  
 extracting: lenet5_data/test_images/img_0037_pred.txt  
  inflating: lenet5_data/test_images/img_0036.hex  
 extracting: lenet5_data/test_images/img_0189_pred.txt  
 extracting: lenet5_data/test_images/img_0061_fc5.hex 

 extracting: lenet5_data/test_images/img_0180_pred.txt  
  inflating: lenet5_data/test_images/img_0032.hex  
  inflating: lenet5_data/test_images/img_0082.hex  
  inflating: lenet5_data/test_images/img_0016.hex  
  inflating: lenet5_data/test_images/img_0021.hex  
 extracting: lenet5_data/test_images/img_0058_label.txt  
 extracting: lenet5_data/test_images/img_0162_label.txt  
 extracting: lenet5_data/test_images/img_0148_fc5.hex  
  inflating: lenet5_data/test_images/img_0187.hex  
 extracting: lenet5_data/test_images/img_0025_pred.txt  
 extracting: lenet5_data/test_images/img_0142_pred.txt  
 extracting: lenet5_data/test_images/img_0120_fc5.hex  
 extracting: lenet5_data/test_images/img_0121_label.txt  
 extracting: lenet5_data/test_images/img_0028_pred.txt  
 extracting: lenet5_data/test_images/img_0023_pred.txt  
  inflating: lenet5_data/test_images/img_0144.hex  
 extracting: lenet5_data/test_images/img_0018_fc5.hex  
  inflating: lenet5_data/test_images/img_0188.hex  
  inflati

  inflating: lenet5_data/test_images/img_0156_fc5.hex  
  inflating: lenet5_data/test_images/img_0105.hex  
 extracting: lenet5_data/test_images/img_0103_fc5.hex  
  inflating: lenet5_data/test_images/img_0044.hex  
 extracting: lenet5_data/test_images/img_0028_label.txt  
 extracting: lenet5_data/test_images/img_0135_label.txt  
 extracting: lenet5_data/test_images/img_0170_label.txt  
  inflating: lenet5_data/test_images/img_0127.hex  
  inflating: lenet5_data/test_images/img_0046_fc5.hex  
 extracting: lenet5_data/test_images/img_0143_pred.txt  
 extracting: lenet5_data/test_images/img_0176_label.txt  
 extracting: lenet5_data/test_images/img_0089_fc5.hex  
  inflating: lenet5_data/test_images/img_0015.hex  
  inflating: lenet5_data/test_images/img_0078.hex  
 extracting: lenet5_data/test_images/img_0086_fc5.hex  
 extracting: lenet5_data/test_images/img_0064_pred.txt  
 extracting: lenet5_data/test_images/img_0009_fc5.hex  
  inflating: lenet5_data/test_images/img_0074.hex  
  infl

 extracting: lenet5_data/test_images/img_0105_fc5.hex  
  inflating: lenet5_data/test_images/img_0053.hex  
 extracting: lenet5_data/test_images/img_0016_pred.txt  
 extracting: lenet5_data/test_images/img_0010_label.txt  
 extracting: lenet5_data/test_images/img_0000_fc5.hex  
  inflating: lenet5_data/test_images/img_0025.hex  
  inflating: lenet5_data/test_images/img_0104.hex  
 extracting: lenet5_data/test_images/img_0177_pred.txt  
 extracting: lenet5_data/test_images/img_0172_pred.txt  
 extracting: lenet5_data/test_images/img_0125_pred.txt  
  inflating: lenet5_data/test_images/img_0092.hex  
 extracting: lenet5_data/test_images/img_0190_label.txt  
 extracting: lenet5_data/test_images/img_0198_pred.txt  
 extracting: lenet5_data/test_images/img_0095_pred.txt  
  inflating: lenet5_data/test_images/img_0077.hex  
 extracting: lenet5_data/test_images/img_0050_fc5.hex  
  inflating: lenet5_data/test_images/img_0128.hex  
 extracting: lenet5_data/test_images/img_0071_fc5.hex  
  infl

 extracting: lenet5_data/test_images/img_0076_fc5.hex  
 extracting: lenet5_data/test_images/img_0165_label.txt  
 extracting: lenet5_data/test_images/img_0069_fc5.hex  
 extracting: lenet5_data/test_images/img_0011_pred.txt  
  inflating: lenet5_data/test_images/img_0009.hex  
  inflating: lenet5_data/test_images/img_0023_fc5.hex  
 extracting: lenet5_data/test_images/img_0170_pred.txt  
 extracting: lenet5_data/test_images/img_0188_pred.txt  
 extracting: lenet5_data/test_images/img_0071_pred.txt  
 extracting: lenet5_data/test_images/img_0114_pred.txt  
 extracting: lenet5_data/test_images/img_0191_pred.txt  
  inflating: lenet5_data/test_images/img_0080.hex  
 extracting: lenet5_data/test_images/img_0130_pred.txt  
  inflating: lenet5_data/test_images/img_0005.hex  
 extracting: lenet5_data/test_images/img_0185_fc5.hex  
  inflating: lenet5_data/test_images/img_0195_fc5.hex  
 extracting: lenet5_data/test_images/img_0042_pred.txt  
 extracting: lenet5_data/test_images/img_0127_pred

In [6]:
# ====================================================================
# 3. 从 npz 加载量化参数，构建 CIMModel
# ====================================================================
DATA_DIR = "lenet5_data"
d = np.load(f"{DATA_DIR}/lenet5_qparams.npz")

model = CIMModel(drv)

# Conv1(1->6, 5x5) -> Pool1(2x2) -> Conv2(6->16, 5x5) -> Pool2(2x2)
# -> FC3(256->120) -> FC4(120->84) -> FC5(84->10)
model.add_conv(d["conv1_weight"], d["conv1_bias"],
               zp=int(d["conv1_zp"]), mult=int(d["conv1_mult"]),
               shift=int(d["conv1_shift"]), stride=1, padding=0, relu=True)
model.add_pool(2, 2)

model.add_conv(d["conv2_weight"], d["conv2_bias"],
               zp=int(d["conv2_zp"]), mult=int(d["conv2_mult"]),
               shift=int(d["conv2_shift"]), stride=1, padding=0, relu=True)
model.add_pool(2, 2)

model.add_fc(256, 120, weight_to_chunks(d["fc3_weight"]), bias_to_u32(d["fc3_bias"]),
             zp=int(d["fc3_zp"]), mult=int(d["fc3_mult"]), shift=int(d["fc3_shift"]),
             relu=True, weight_int8=d["fc3_weight"], bias_int32=d["fc3_bias"])

model.add_fc(120, 84, weight_to_chunks(d["fc4_weight"]), bias_to_u32(d["fc4_bias"]),
             zp=int(d["fc4_zp"]), mult=int(d["fc4_mult"]), shift=int(d["fc4_shift"]),
             relu=True, weight_int8=d["fc4_weight"], bias_int32=d["fc4_bias"])

model.add_fc(84, 10, weight_to_chunks(d["fc5_weight"]), bias_to_u32(d["fc5_bias"]),
             zp=int(d["fc5_zp"]), mult=int(d["fc5_mult"]), shift=int(d["fc5_shift"]),
             relu=False, weight_int8=d["fc5_weight"], bias_int32=d["fc5_bias"])

# Show packing info
for i, layer in enumerate(model.layers):
    if layer.get("type") == "conv":
        p = layer.get("_packed")
        k = p["k_pack"] if p else 1
        print(f"  Layer {i} ({layer['C_in']}ch {layer['K_h']}x{layer['K_w']}->{layer['C_out']}ch): "
              f"col_len={layer['col_len']}, k_pack={k}")

print(f"\nModel: {len(model.layers)} layers loaded from npz")

  Layer 0 (1ch 5x5->6ch): col_len=25, k_pack=21
  Layer 2 (6ch 5x5->16ch): col_len=150, k_pack=5

Model: 7 layers loaded from npz


In [7]:
# ====================================================================
# 4. 单图推理 + bit-exact 验证 + profiler
# ====================================================================
def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

img_dir = f"{DATA_DIR}/test_images"
img_path = sorted(glob.glob(f"{img_dir}/img_????.hex"))[0]
img_u8 = np.array(read_hex_u8(img_path), dtype=np.uint8)
label = int(open(img_path.replace(".hex", "_label.txt")).read().strip())
print(f"Test image: {os.path.basename(img_path)}, label={label}\n")

print("=== Single Image LeNet-5 (verify=True, profile=True) ===")
pred, logits, prof = model.predict(
    img_u8.reshape(1, 28, 28),
    verbose=True, verify=True, profile=True,
)

print(f"\nHW pred={pred}, label={label}, {'CORRECT' if pred == label else 'WRONG'}")
print(f"\n--- Profiler ---")
print(f"{'Layer':<30s} {'n_mvm':>6s} {'k_pack':>6s} {'setup':>8s} {'load_x':>8s} "
      f"{'compute':>8s} {'read':>8s} {'total':>8s}")
for lp in prof["layers"]:
    if lp["type"] == "pool":
        print(f"{lp['name']:<30s} {'--':>6s} {'--':>6s} {'--':>8s} {'--':>8s} "
              f"{'--':>8s} {'--':>8s} {lp['total_ms']:>7.1f}ms")
    else:
        print(f"{lp['name']:<30s} {lp['n_mvm']:>6d} {lp['k_pack']:>6d} "
              f"{lp.get('setup_ms',0):>7.1f}ms {lp.get('load_x_ms',0):>7.1f}ms "
              f"{lp.get('compute_ms',0):>7.1f}ms {lp.get('read_out_ms',0):>7.1f}ms "
              f"{lp['total_ms']:>7.1f}ms")
print(f"{'TOTAL':<30s} {'':>6s} {'':>6s} {'':>8s} {'':>8s} {'':>8s} {'':>8s} "
      f"{prof['total_ms']:>7.1f}ms")

Test image: img_0000.hex, label=7

=== Single Image LeNet-5 (verify=True, profile=True) ===
[verify] run_id = no-git_20221022_102443
  Layer 0 (Conv 1ch 5x5->6ch): 28 packed MVMs (k_pack=21), 66304 cycles
  Conv: feat[1,28,28] * weight[6,1,5,5]  stride=1 pad=0  mode=explicit
  Output spatial: 24×24 = 576 pixels, col_len=25, MVMs=576
  [MATCH]   layer_0 (conv_1x5x5_to_6)  3456 elements
  Layer 1 (Pool 2x2): -> (6, 12, 12)
  Layer 2 (Conv 6ch 5x5->16ch): 13 packed MVMs (k_pack=5), 24700 cycles
  Conv: feat[6,12,12] * weight[16,6,5,5]  stride=1 pad=0  mode=explicit
  Output spatial: 8×8 = 64 pixels, col_len=150, MVMs=64
  [MATCH]   layer_2 (conv_6x5x5_to_16)  1024 elements
  Layer 3 (Pool 2x2): -> (16, 4, 4)
  Layer 4 (FC 256->120): 1552 cycles
  [MATCH]   layer_4 (fc_256x120)  120 elements
  Layer 5 (FC 120->84): 876 cycles
  [MATCH]   layer_5 (fc_120x84)  84 elements
  Layer 6 (FC 84->10): 134 cycles
  [MATCH]   layer_6 (fc_84x10)  10 elements

HW pred=7, label=7, CORRECT

--- Profiler 

In [8]:
# ====================================================================
# 5. 批量推理：20 张图，统计正确率 + 平均耗时
# ====================================================================
img_files = sorted(glob.glob(f"{img_dir}/img_????.hex"))
n_images = len(img_files)
print(f"=== Batch Test: {n_images} images ===\n")

correct = 0
t0 = time.time()

for img_path in img_files:
    name = os.path.basename(img_path).replace(".hex", "")
    img = np.array(read_hex_u8(img_path), dtype=np.uint8)
    label = int(open(f"{img_dir}/{name}_label.txt").read().strip())

    pred, logits = model.predict(img.reshape(1, 28, 28))

    if pred == label:
        correct += 1
    else:
        print(f"  {name}: pred={pred} label={label} WRONG")

elapsed = time.time() - t0
print(f"\nResults: {correct}/{n_images} correct ({100*correct/n_images:.1f}%)")
print(f"Wall time: {elapsed:.2f}s ({elapsed/n_images*1000:.1f} ms/image)")

=== Batch Test: 200 images ===

  img_0018: pred=8 label=3 WRONG

Results: 199/200 correct (99.5%)
Wall time: 342.96s (1714.8 ms/image)
